# Notes

* Notebooks are in JSON format - you can use `json.load(file)` to load the data into a dictionary
* Dictionary format
    * Top-level keys: `dict_keys(['cells', 'metadata', 'nbformat', 'nbformat_minor'])`
    * `data["cells"]` is a list of notebook cells, which are either Markdown or code. In Markdown, there can be HTML. Should I reformat the HTML?
        * Each cell is a dictionary. The only keys you need to care about are `"cell_type"` and `"source"`.
        * `source` contains the text as a list of strings (I think each element is one line)
    * `data["metadata"]["language info"]["name"]` contains the language (e.g. Python)
    * Should you reformat everything into Markdown? Code cells - delimit with triple backticks: ```python {code}```
* Clean up images - the link can take up a lot of space. For example, see Exercise_Solution_LLM_Data_Prep.md
* Cleaning up HTML images (or other elements) - I think this code can be specifically for notebooks. HTMLToDocument uses trafilatura for cleanup. 
    * I should use BeautifulSoup for more control over what's cleaned up. Seems like I can just use soup.get_text() for this. BeautifulSoup.get_text() removes list formatting, which imo is not desirable.
    * If this code is specifically for notebooks, then I probably don't need to modularize it
* Cleaning up markdown images
    * Regex
    * Haystack Markdown converter removes everything but the text - I wonder if this is good. There's no more header information, triple backticks to delimit code, etc. It does seem to remove HTML images.
* Code blocks - if there's nothing but white space in the code block, skip it
* HTML/Markdown formatting elements may be helpful to the LLM to understand the material - e.g. headers, lists, etc. PPTX, PDF conversion - be careful not to lose structure
* For chunking, there is already a natural structure that would lend itself well to recursive chunking
    * Group all adjacent code cells + 1 markdown cell
    * Markdown - group based on header (#)

In [ ]:
import glob
from pymilvus import MilvusClient
from dotenv import load_dotenv
import os
import pandas as pd
import pickle
import re

load_dotenv(override=True)

files = glob.glob("Content/**/*.ipynb", recursive=True)
files

['Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/_m3.1-data-prep-text-lab-1.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Demo_LLM_Data_Prep_101.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Demo_Text_Data_Augmentation_Synthetic_Data.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Exercise_Solution_Data_Augmentation_GPT.ipynb',
 'Content/La

In [ ]:
import json

file = files[0]

with open(file, 'r') as f:
    data = json.load(f)

In [45]:
file

'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb'

In [14]:
data.keys()

dict_keys(['cells', 'metadata', 'nbformat', 'nbformat_minor'])

In [15]:
type(data["cells"])

list

In [17]:
len(data["cells"])

28

In [18]:
data["cells"][0]

{'cell_type': 'markdown',
 'metadata': {'id': 'KfqyGY5kUSy2'},
 'source': ['<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">\n',
  '<h1><center>ArXiv Data Cleaning Notebook</center></h1>']}

In [21]:
[cell["cell_type"] for cell in data["cells"]]
[f'{cell["cell_type"]}: {cell.keys()}' for cell in data["cells"]]

["markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "code: dict_keys(['cell_type', 'execution_count', 'metadata', 'outputs', 'source'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "code: dict_keys(['cell_type', 'execution_count', 'metadata

In [33]:
for cell in data["cells"]:
    if cell["cell_type"] == "markdown":
        # print(f'{cell["source"]}\n{"-"*20}\n')
        print(type(cell["source"]))

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


In [32]:
for cell in data["cells"]:
    if cell["cell_type"] == "code":
        # print(f'{cell["source"]}\n{"-"*20}\n')
        print(type(cell["source"]))

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


In [30]:
for cell in data["cells"]:
    if cell["cell_type"] == "markdown":
        print(f'{cell["metadata"]}\n{"-"*20}\n')

{'id': 'KfqyGY5kUSy2'}
--------------------

{'id': '-tzs7zPKUSy7'}
--------------------

{'id': 'j-YdiN4u4v1C'}
--------------------

{'id': 'srWsJFlH4v1O'}
--------------------

{'id': 'dZaYV0_oliOT'}
--------------------

{'id': 'H_tESoKilqwH'}
--------------------



In [34]:
for cell in data["cells"]:
    if cell["cell_type"] == "markdown":
        # print(f'{cell["source"]}')
        for elem in cell["source"]:
            print(elem)
        print("\n")
    elif cell["cell_type"] == "code":
        print("```python")
        # print(f'{cell["source"]}')
        for elem in cell["source"]:
            print(elem)
        print("```")
        print("\n")

<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

<h1><center>ArXiv Data Cleaning Notebook</center></h1>


## Introduction



Welcome to the ArXiv Data Cleaning Notebook! Here, we will walk you through the process of preparing and cleaning ArXiv data using Python methods.



 - **Cleaning data with scripts**: You can clean data using scripts and EC2 accounts.



 - **Manually Cleaning data**: Dive deep into the basics and clean ArXiv datasets from scratch, offering a glimpse into the essence of use of Large Language Model.


```python
# Here is a repository we use in this lab.

!git clone https://github.com/togethercomputer/RedPajama-Data.git

%cd /content/RedPajama-Data/data_prep/arxiv
```


## Cleaning data with scripts



Follow these instructions to create the Arxiv dataset. These steps assume that you are in the data_prep/arxiv directory.


```python
# Setup

# Install the dependencies specified in arxiv_requirements.txt:

!pip install -

In [44]:
from copy import deepcopy
fname = "ipynb_sample.md"

all_cells = []
for cell in data["cells"]:

    content = deepcopy(cell["source"])
    if cell["cell_type"] == "code":
        content.insert(0, "\n```python\n")
        content.append("\n```\n")
    
    all_cells.extend(content)

writestr = "".join(all_cells)
print(writestr)

with open(fname, "w") as f:
    f.write(writestr)

<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">
<h1><center>ArXiv Data Cleaning Notebook</center></h1>## Introduction

Welcome to the ArXiv Data Cleaning Notebook! Here, we will walk you through the process of preparing and cleaning ArXiv data using Python methods.

 - **Cleaning data with scripts**: You can clean data using scripts and EC2 accounts.

 - **Manually Cleaning data**: Dive deep into the basics and clean ArXiv datasets from scratch, offering a glimpse into the essence of use of Large Language Model.
```python
# Here is a repository we use in this lab.
!git clone https://github.com/togethercomputer/RedPajama-Data.git
%cd /content/RedPajama-Data/data_prep/arxiv
```
## Cleaning data with scripts

Follow these instructions to create the Arxiv dataset. These steps assume that you are in the data_prep/arxiv directory.
```python
# Setup
# Install the dependencies specified in arxiv_requirements.txt:
!pip install -r arxiv_requirements.t

In [38]:
mylist = [1, 2, 3]

mylist.insert(0,0)

mylist.append(4)

mylist

[0, 1, 2, 3, 4]

# Get notebook kernel spec

In [98]:
import glob
from dotenv import load_dotenv
import os
import pandas as pd
import json

load_dotenv(override=True)

files = glob.glob("Content/**/*.ipynb", recursive=True)
# files

for file in files:
    with open(file, 'r') as f:
        data = json.load(f)
    
    # print(file)
    # print(data["metadata"])
    # print(data["metadata"]["kernelspec"].keys())
    # print(f'display_name: {data["metadata"]["kernelspec"].get("display_name", "none")}, name: {data["metadata"]["kernelspec"].get("name", "none")}, language: {data["metadata"]["kernelspec"].get("language", "none")}')

    # print(data["metadata"]["kernelspec"].get("language", data["metadata"]["kernelspec"].get("name", data["metadata"]["kernelspec"].get("display_name", ""))))
    print(data["metadata"]["kernelspec"].get("language") or data["metadata"]["kernelspec"].get("name") or data["metadata"]["kernelspec"].get("display_name") or "")
    # display_name
    # name
    # language
    print("-"*100)

python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python3
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
--------------------

# Remove image links

In [194]:
import re

html_img = 'asdfasdfasdf <img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%"> asdfasjdf;ajkg9224'
html_img = '<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">\
<h1>Exercise: Processing Data for Large Language Models</h1>\
Developed by WeCloudData\
<br></br>'

pattern = re.compile(r'<img src=.*?>')
pattern = re.compile(r'<img[^>]*?>')
pattern = re.compile(r'<img[^>]*?>|!\[.*?\]\(.*?\)', flags=re.DOTALL)
# re.compile(r'!\[.*?\]\(.*?\)')

m = pattern.search(html_img)
m.group()

'<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">'

In [186]:
pattern.sub("", html_img)

'<h1>Exercise: Processing Data for Large Language Models</h1>Developed by WeCloudData<br></br>'

In [200]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_img, "html.parser")
# soup = BeautifulSoup(html_img)
# soup = BeautifulSoup(md_img, "html.parser")
print(soup.prettify())
print(soup.get_text())
matches = soup.find_all(["img"])
for match in matches:
    match.decompose()
str(soup)

<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%"/>
<h1>
 Exercise: Processing Data for Large Language Models
</h1>
Developed by WeCloudData
<br/>

Exercise: Processing Data for Large Language ModelsDeveloped by WeCloudData


'<h1>Exercise: Processing Data for Large Language Models</h1>Developed by WeCloudData<br/>'

In [162]:
md_img = '## ![Capture.PNG](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAA4YAAAJYCAYAAAA+MPjmAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAJcEhZcwAADsMAAA7DAcdvqGQAAMnhSURBVHhe7f1/tGXJVd8Jptvu7jW9utf80a3l1curGzMslsd4Zmxst8Bu9djGNJiRZYsHCQY8bRswGIwkoCgsLLDs9ENyg4Wr2pmSxY+yhOalQBIGZeknppQCocxqLCWUSlVColJCKklVylL9kqpUv3Umfp6z946Ic85979737o/PR2ur7jkRsWPHjrjnxjfPefee6uBYeezxp7vPPvgYhmEYhmEYhmHY2hjC8BiZEoX3O/v8o092Tzz5TPf0M1/svujaePOv/Tlf5uvU2h7aHnqsu++BL3T3heMvlOUYhmEYhmEYhm29IQyPiSefeqY6Ad7ud+LMi8a5+Lq+Tc3XXAvtHyjPYxiGYRiGYRi2e4YwPAaeeeaLzTt9Dz/yRLgruCi+zedc25rPKbv/wccRhRiGYRiGYRiG9YYwPAYe+nxdwH3hsadSjcPzqPNR841hGIZhGIZhGDbXEIYr5qmn64+Q+r8XXBbeV62Pmh31EVQMwzAMwzAMw7bPdkoYfvrTn+5uu+227hOf+ET3xS8OD3Def//94fwXvvCFdGZ51B73fPjzT6TS5eF92n4wDMMwDMMwDMPm2MqF4R133NF98IMfTEcnw9WrV7uXvvSl3dd//df39q3f+q3dwcFB9+STT3Zvf/vbw7m77rortVgOXnrWku6/YbSGz9XP/uzPdn/wB3+QzszH+6z1hWEYhmEYhmEYNmUrFYZPPfVU97a3va174xvf2N1+++3p7PHykY98pPvbf/tvB+H3oz/6o92rXvWq7t/8m3/T/YN/8A/Cue/93u/tfuEXfmElwvCJJ8ufp/CPctb4lV/5le5LvuRLgv3JP/knu3e+852pZD6zHhOd8aUzv//67+y+5BWXq2UbZb95Y/clf/9N3e/XyjAMwzAMwzAM623ldwz945lvfetbgzj0j2seJw899FD3bd/2bd03fdM3db/zO7+Tzg74eJ7//OcHUbgKYfi5R8q//Xuk8beFX/M1X9N953d+Z/fZz362+1t/62913/zN3xzOf/mXf3kvGLP9hb/wF0KZxfu2/R3GjksYLrufi6/4ku7vvf4TwzmEIYZhGIZhGIbNsmP5G8OTEoc/8zM/EwTfxYsX05nIE0880f3iL/5isB/8wR9cmTB88HOPFwl/6qn6Y6Tf+I3fGMThb//2bwfh9z3f8z3h/Ld/+7d33/It36LsH/7DfxjKLGO/lbiIIQwxDMMwDMMwbLfs2L58RorDK1eupLOr45lnngmPkGaBJfn85z/fi0FpyxaGtUc7xXfeKHxO/tyf+3PhjuCzn/3sQ/2dofdt+yvtE93B3/+S7qd/M/4334X86d8c6mjBput9yZfc2F3sfV3ufvpLvrM7+M03dX+vWp7rDO2zcPMibvCZ+0/+/kC28cfRVyH8sp9XvN3EmNokYXjRjyeXWSHq6/TtTPmc9tLGfOWxvT7V6cvMWEMubQ4xDMMwDMMwbLW2NGHo/57Q35kbs3e84x1BGHr7vd/7vdRyNfhvHvVi71//63+dzhw/tYSP8fjjj3cf+MAHQi4PS61PbVnoDYLrs3/gxchwrIShK/vpXozFtoM4S6KmvyuXfBvRM4jOJI5q/Yj6SlwGsZWOk1Ab7gBqf9U7hs5ffy6MU8QTykUe7Pim2kub8pXHVhnv4C/PDcIQwzAMwzAMO17bWmH4yU9+MgjDn/7pn05n2txyyy3hsdLHHhsXbotyfyXhh8Xn13+Bj38MtsVidwz1eSmqxh7x1GVamAWT4k2+Tubbt/up+FPxmtiN/6owNP3LOkV9bxPxV9u0zqv2E7nKFsQnwhDDMAzDMAw7XtvaR0n9z1D8zb/5N6uPkkr8I6d7e3vd8573PPXbhsvgwYfLvzF85pnF+/CPuP7lv/yXw92rr/zKr2zm72nn2/ZXWl0Yjgk2L3rio47JZgrD4Ee2M+3nCcPYf45XtpHn87ESZ6PCrp4HJcxmC8MZvipjK8fvDGGIYRiGYRiGnYBt9ZfP/NRP/VS4a/ibv/mb6UyJ/y1DX+cVr3hFOrM8Hqr86HzrNwy/67u+q/iSme/4ju8IZf4nNZ7znOeEHP6Nv/E3whfV1Jj3W4Z1ESMFjxVfUghpMTNxF8y/tsJH2DxhaOLthZOvqwXUYsKwIfK8fxn/LGE4w1dLGBr/oU+EIYZhGIZhGHbMtnJheJI/V/GHf/iH4eco/M9V/Kf/9J/S2Yh/bPQ1r3lNEIX+5yGuXbuWSpbHI1+o/FyFO1fjz/7ZP1vcWfM/VeHxdzS9GHz66aeDgPQisca8n6uIQksJEiNGBsFmRWRqO1cYhvLKnbRkdWEo/dfFUxBhf3+4w6nOy3NTwi6MW8Yfx6fKR9r74uiT6n4snCzdve+YNS5KoyKwa9uXOXff9iLDKe9A2tVgxmvzI22V8tzwC7SxaH2ib/ng52j/7v1kbuxtm/bVvlWsri0Nha3uWekzsAAFg7jlEY7hJetC1bkEURmcUmAAAAAADAskAYLoNwZ6/xaOuhMeJSPXYKAAAAAACwPBCGS2IVj2/ax165WwgAAAAAAKsAYQgAAAAAALDjIAwBAAAAAAB2HIQhAAAAAADAjrORwvD+Czd0z7nuYnd/Os60zs/m3ovd9c+9obt5wa/Xvv3cS7rnnLvDvbrW3XzdS7rrL1yLBQAAAAAAABsAwlCCMAQAAAAAgJ2j6/7/oIbCOIq2UH0AAAAASUVORK5CYII=)'

In [163]:
pattern = re.compile(r'!\[.*?\]\(.*?\)')
pattern = re.compile(r'<img src=.*?>|!\[.*?\]\(.*?\)', flags=re.DOTALL)
pattern = re.compile(r'<img[^>]*?>|!\[.*?\]\(.*?\)', flags=re.DOTALL)

m = pattern.search(md_img)
pattern.sub("", md_img)

'## '

# Clean all notebooks

In [180]:
import glob
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv(override=True)

files = glob.glob("Content/**/*.ipynb", recursive=True)
files

['Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/_m3.1-data-prep-text-lab-1.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Demo_LLM_Data_Prep_101.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Demo_Text_Data_Augmentation_Synthetic_Data.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Exercise_Solution_Data_Augmentation_GPT.ipynb',
 'Content/La

In [201]:
import json

# class NotebookParser:
#     def __init__(self):
#         self.remove_pattern = re.compile(r'<img[^>]*?>|!\[.*?\]\(.*?\)', flags=re.DOTALL)

#     def parse_notebook(self, filename: str) -> str:
#         with open(filename, 'r') as f:
#             data = json.load(f)

#         kernelspec = data.get("metadata", {}).get("kernelspec", {})
#         lang = kernelspec.get("language") or kernelspec.get("name") or kernelspec.get("display_name") or ""

#         all_cells = []
#         for cell in data["cells"]:
#             content = cell.get("source", [])

#             if cell["cell_type"] == "code":
#                 all_cells.append(f"\n```{lang}\n")
#                 all_cells.extend(content)
#                 all_cells.append("\n```\n")
#             elif cell["cell_type"] == "markdown":
#                 content = [self.remove_pattern.sub("", text) for text in content]
#                 all_cells.append("  \n")
#                 all_cells.extend(content)
#                 all_cells.append("  \n")
#             else:
#                 all_cells.append("  \n")
#                 all_cells.extend(content)
#                 all_cells.append("  \n")

#         return "".join(all_cells)

from bs4 import BeautifulSoup

# class NotebookParser:
#     def __init__(self):
#         self.remove_pattern = re.compile(r'!\[.*?\]\(.*?\)', flags=re.DOTALL)

#     def parse_notebook(self, filename: str) -> str:
#         # Update with Path
#         with open(filename, 'r') as f:
#             data = json.load(f)

#         kernelspec = data.get("metadata", {}).get("kernelspec", {})
#         lang = kernelspec.get("language") or kernelspec.get("name") or kernelspec.get("display_name") or ""

#         all_cells = []
#         for cell in data["cells"]:
#             content = "".join(cell.get("source", [])).strip()
#             if content == "": # ignore the cell if it's entirely whitespace (empty)
#                 continue

#             if cell["cell_type"] == "code":
#                 composed_str = f"\n```{lang}\n{content}\n```\n"
#             elif cell["cell_type"] == "markdown":
#                 content = self.remove_pattern.sub("", content) # clean up using regex
#                 content = BeautifulSoup(content, "html.parser").get_text() # clean up HTML
#                 composed_str = f"  \n{content}  \n"
#             else:
#                 composed_str = f"  \n{content}  \n"

#             all_cells.append(composed_str)

#         return "".join(all_cells)

class NotebookParser:
    def __init__(self):
        self.remove_pattern = re.compile(r'!\[.*?\]\(.*?\)', flags=re.DOTALL) # remove Markdown images

    def parse_notebook(self, filename: str) -> str:
        # Update with Path
        with open(filename, 'r') as f:
            data = json.load(f)

        kernelspec = data.get("metadata", {}).get("kernelspec", {})
        lang = kernelspec.get("language") or kernelspec.get("name") or kernelspec.get("display_name") or ""

        all_cells = []
        for cell in data["cells"]:
            content = "".join(cell.get("source", [])).strip()
            if content == "": # ignore the cell if it's entirely whitespace (empty)
                continue

            if cell["cell_type"] == "code":
                composed_str = f"\n```{lang}\n{content}\n```\n"
            elif cell["cell_type"] == "markdown":
                content = self.remove_pattern.sub("", content) # clean up using regex
                # Remove HTML images
                content = BeautifulSoup(content, "html.parser")
                for match in content.find_all(["img"]):
                    match.decompose()
                content = str(content)
                composed_str = f"  \n{content}  \n"
            else:
                composed_str = f"  \n{content}  \n"

            all_cells.append(composed_str)

        return "".join(all_cells)

In [202]:
import re
os.makedirs("./converted_notebooks/", exist_ok=True)

nb_parser = NotebookParser()

for file in files:
    _, tail = os.path.split(file)
    converted_filename = re.sub(".ipynb", ".md", tail)
    
    converted_notebook = nb_parser.parse_notebook(file)
    with open(f"./converted_notebooks/{converted_filename}", "w") as f:
        f.write(converted_notebook)
        print(converted_filename)

Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).md
Exercise_LLM_Data_Prep.md
Exercise_Solution_LLM_Data_Prep.md
_m3.1-data-prep-text-lab-1.md
Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.md
Demo_LLM_Data_Prep_101.md
Demo_Text_Data_Augmentation_Synthetic_Data.md
Exercise_Solution_Data_Augmentation_GPT.md
Exercise_Solution_Data_Augmentation_with_BackTranslation.md
Exercise_Web_Scraping_Selenium_Linkedin.md
Exercise_Solution_Web_Scraping_BS4.md
Exercise_Solution_Web_Scraping_Selenium_Linkedin.md
Exercise_Web_Scraping_BS4.md
Demo_Web_Scraping_HTML_Xpath.md
Exercise_Solution_LLM_Annotation_Sagemaker_Groundtruth.md
Exercise_Solution_Data_Cleaning_Cleanlab_Bank_Intent.md
Exercise_Solution_Rule-based_Automatic_Data_Labeling.md
Exercise_Solution_Data_Labeling_Issue_Detection_Cleanlab_TwitterData.md
Demo_Data_Labelling_Cleanlab_Identify_Label_Issues.md
Demo_LLM-based_Labelling.md
Exercise_Data_Cleaning_Cleanlab_Bank_Intent.md
Demo_RAG_2_LangChain_Splitter.md
Demo_RAG_1_LangChain_FAISS.md
Ex

In [73]:
file

'Content/Data wrangling with Python/Chapter 9. Groupby Mechanics/Exercise_Part_5_Grouping with Functions.ipynb'

In [75]:
import re

obj = re.search(r"\.\S+$", file)

In [80]:
obj.group(0)

'.ipynb'

In [ ]:
obj = re.compile(r"\.[a-zA-Z0-9]+$")
m = obj.search(file)
m.group()

'.ipynb'

In [91]:
try:
    obj.search(None)
except TypeError:
    pass

In [88]:
obj.search(None)

TypeError: expected string or bytes-like object, got 'NoneType'

In [102]:
d = {"a": 1}

print(d.get("b", {}).get("a"))

None


In [106]:
print(d.get("b") or d.get("c"))

None


In [129]:
d = {"a": ["a", "b"]}

In [130]:
l = d.get("a").copy()

In [131]:
l[0] = "c"
l

['c', 'b']

In [132]:
d.get("a")

['a', 'b']